# Installation of necessary libraries

In [3]:
# !pip install jupyter_contrib_nbextensions

In [4]:
# !jupyter contrib nbextension install --user

In [5]:
# pip install xgboost

In [6]:
# pip install --upgrade pip

In [7]:
# pip install LightGBM

In [8]:
# pip install --upgrade lightgbm

In [9]:
# pip install SHAP

In [10]:
# pip install ipywidgets

In [11]:
# !pip3 install mlflow

In [12]:
# !jupyter nbextension enable toc2/main

# Import Libraries

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from scipy.stats import norm

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import shap
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
import lightgbm
from sklearn.ensemble import AdaBoostClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import pickle
import mlflow
import os

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
df = pd.read_csv('Network_anomaly_data org.csv')

In [14]:
df.head()

NameError: name 'df' is not defined

In [ ]:
df.info()

# Data Cleaning

## Null Value Check

- **No missing values found in the dataset**

In [ ]:
df.isnull().sum()

## Duplicate Value Check
- **No dupliacte records present in the dataset**

In [ ]:
df.duplicated().sum()

## Outlier Detection
- **There are outliers present in the data as per the large deviations between the 75% and 100% values in the statistical analysis.**
- **As per the box also, outliers are present in the data.**
- **However, these outliers consist of important information that help in identifying the type of connection, i.e. attack vs normal and therefore will not be removed from this dataset.**

In [ ]:
df.describe().T

In [ ]:
plt.figure(figsize=(20, 40))
df.plot(kind='box', subplots=True, layout=(8, 5), figsize=(20, 40))
plt.show()

# Feature Engineering

## Type - Normal vs Attack

In [ ]:
def type(x):
  if x == 'normal':
    return 'normal'
  else:
    return 'attack'

In [ ]:
df['type'] = df['attack'].apply(type)

In [ ]:
df.type.value_counts()

In [ ]:
df.attack.value_counts()

## Category - DoS, Probe, U2R, R2L

In [ ]:
df.attack.unique()

In [ ]:
def attack_category(x):
  if x in ['neptune', 'teardrop', 'smurf', 'pod', 'back', 'land']:
    return 'DoS'
  elif x in ['rootkit', 'buffer_overflow', 'multihop', 'loadmodule', 'perl']:
    return 'U2R'
  elif x in ['ipsweep', 'nmap', 'portsweep', 'satan']:
    return 'Probe'
  elif x in ['guess_passwd', 'warezmaster', 'warezclient', 'imap', 'phf', 'spy', 'ftp_write']:
    return 'R2L'
  else:
    return 'normal'

In [ ]:
df['attack_category']=df['attack'].apply(attack_category)

In [ ]:
df.attack_category.value_counts()

In [ ]:
def pie_plot(df, cols_list, rows, cols):
    fig, axes = plt.subplots(rows, cols)
    for ax, col in zip(axes.ravel(), cols_list):
        df[col].value_counts().plot(ax=ax, kind='pie', figsize=(15, 15), fontsize=10, autopct='%1.0f%%')
        ax.set_title(str(col), fontsize = 12)
    plt.show()

In [ ]:
df['attack_distribution']=df[df['attack_category']!='normal']['attack_category']

In [ ]:
pie_plot(df, ['type', 'attack_distribution'], 1, 2)

**Pie Chart Insights**

- The data consists of almost **equal** distribution of normal vs attack records.
- **DoS** followed by **Probe** are the major attack types present in the dataset.

In [ ]:
df.drop(['attack_distribution'], axis=1, inplace=True)

# EDA - Visualization

## Extract Numerical and Categorical Columns

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

## Univariate Analysis

In [ ]:
categorical_cols = ['protocoltype', 'flag', 'service']

for col in categorical_cols:
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df, x=col, hue='attack_category')
    plt.title(f'Distribution of {col} by Attack Category')
    plt.xticks(rotation=90)
    plt.show()

**Bar Chart Insights:**
- It can be observed that the most used protocol is **tcp** and the most used flag and service are **SF** and **http** respectively for the normal traffic.
- It can also be observed that **DoS** attacks are mainly targeted on **private** network.

## Bivariate Analysis
- There seems not much correlation between the features as observed for the scatterplots below.

In [ ]:
scatter_cols = ['srcbytes', 'dstbytes', 'duration', 'count']

sns.pairplot(df, vars=scatter_cols, hue='attack_category', diag_kind='kde')
plt.suptitle('Scatter Plot Matrix for Numerical Columns', y=1.02)
plt.show()

## Multivariate Analysis

### Label Encoding

In [ ]:
# Duplicate dataset before label encoding for hypothesis testing
df_hyp = df.copy()

In [ ]:
# Dictionary to store mappings
label_mappings = {}
# Label Encoding
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    
    # Store the mapping in the dictionary
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    
# Display the mappings
for col, mapping in label_mappings.items():
    print(f"Label encoding mapping for {col}: {mapping}")

### Heatmap

In [15]:
# Increase the figure size
plt.figure(figsize=(20, 16))

# Generate a mask for the upper triangle (optional, if you want to hide the upper triangle)
mask = np.triu(np.ones_like(df.corr(), dtype=bool))

# Create the heatmap
sns.heatmap(df.corr(), mask=mask, annot=True, fmt=".2f", cmap='coolwarm', cbar_kws={"shrink": .8})

# Rotate the tick labels for better readability
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)

# Add a title (optional)
plt.title('Correlation Heatmap', fontsize=16)

# Display the heatmap
plt.show()

NameError: name 'df' is not defined

<Figure size 2000x1600 with 0 Axes>

In [ ]:
# Compute the correlation matrix
corr_matrix = df.corr().abs()

# Identify upper triangle of the correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.9
to_drop = [column for column in upper.columns if any(upper[column] >= 0.5)]

# Drop features
df_reduced = df.drop(columns=to_drop)

print(f"Dropped columns: {to_drop}")
print(f"Remaining columns: {df_reduced.columns.tolist()}")

#mask
mask_t = np.triu(np.ones_like(df_reduced.corr(), dtype=bool))

# Optionally, display the reduced correlation matrix
plt.figure(figsize=(20, 15))
sns.heatmap(df_reduced.corr(), mask = mask_t, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap After Removing Highly Correlated Variables', size=20)
plt.show()

# Hypothesis Testing

## Network Traffic Volume - Normal vs Attack

In [ ]:
normal = df_hyp[df_hyp['type']=='normal']
anomalous = df_hyp[df_hyp['type']!='normal']

In [ ]:
# Perform t-tests
t_src, p_src = stats.ttest_ind(normal['srcbytes'], anomalous['srcbytes'], equal_var=False)
t_dst, p_dst = stats.ttest_ind(normal['dstbytes'], anomalous['dstbytes'], equal_var=False)

print(f"T-test for Src_bytes: t-statistic = {t_src}, p-value = {p_src}")
print(f"T-test for Dst_bytes: t-statistic = {t_dst}, p-value = {p_dst}")

In [ ]:
# Interpret the results
alpha = 0.05
if p_src < alpha:
    print("Reject the null hypothesis for Src_bytes: Significant difference in means between normal and anomalous connections.")
else:
    print("Fail to reject the null hypothesis for Src_bytes: No significant difference in means between normal and anomalous connections.")

print()

if p_dst < alpha:
    print("Reject the null hypothesis for Dst_bytes: Significant difference in means between normal and anomalous connections.")
else:
    print("Fail to reject the null hypothesis for Dst_bytes: No significant difference in means between normal and anomalous connections.")

## Network Traffic Volume - DoS, Probe, R2L, U2R 

In [ ]:
dos = df_hyp[df_hyp['attack_category']=='DoS']
prb = df_hyp[df_hyp['attack_category']=='Probe']
r2l = df_hyp[df_hyp['attack_category']=='R2L']
u2r = df_hyp[df_hyp['attack_category']=='U2R']

In [ ]:
# ANOVA for src_bytes
anova_src = stats.f_oneway(dos['srcbytes'], prb['srcbytes'], r2l['srcbytes'], u2r['srcbytes'])
print(f"ANOVA for src_bytes: F-statistic = {anova_src.statistic}, p-value = {anova_src.pvalue}")

# ANOVA for dst_bytes
anova_dst = stats.f_oneway(dos['dstbytes'], prb['dstbytes'], r2l['dstbytes'], u2r['dstbytes'])
print(f"ANOVA for dst_bytes: F-statistic = {anova_dst.statistic}, p-value = {anova_dst.pvalue}")

In [ ]:
# Interpret the results
alpha = 0.05
if anova_src.pvalue < alpha:
    print("Reject the null hypothesis for src_bytes: Significant difference in means between attack categories.")
else:
    print("Fail to reject the null hypothesis for src_bytes: No significant difference in means between attack categories.")

print()

if anova_dst.pvalue < alpha:
    print("Reject the null hypothesis for dst_bytes: Significant difference in means between attack categories.")
else:
    print("Fail to reject the null hypothesis for dst_bytes: No significant difference in means between attack categories.")

In [ ]:
# Combine data for Tukey's HSD test
src_bytes_data = df_hyp[['attack_category', 'srcbytes']]
dst_bytes_data = df_hyp[['attack_category', 'dstbytes']]

# Perform Tukey's HSD test for src_bytes
tukey_src_bytes = pairwise_tukeyhsd(endog=src_bytes_data['srcbytes'], groups=src_bytes_data['attack_category'], alpha=0.05)
print(tukey_src_bytes)

# Perform Tukey's HSD test for dst_bytes
tukey_dst_bytes = pairwise_tukeyhsd(endog=dst_bytes_data['dstbytes'], groups=dst_bytes_data['attack_category'], alpha=0.05)
print(tukey_dst_bytes)


Summary of Tukey HSD Test Results:
* Significant Differences:

    * **DoS vs. Probe:** There is a significant difference in means between DoS and Probe attack categories for both comparisons (meandiff: 384503.5172 and 180905.7103, p-adj: 0.0 and 0.0001, respectively).
    * **Probe vs. Normal:** There is a significant difference in means between Probe and normal categories for both comparisons (meandiff: -372546.559 and -176745.2266, p-adj: 0.0 and 0.0001, respectively).

- No significant difference in means was found between other attack categories (DoS vs. R2L, DoS vs. U2R, DoS vs. normal, Probe vs. R2L, Probe vs. U2R, R2L vs. U2R, R2L vs. normal, U2R vs. normal) as indicated by high p-adj values (all > 0.05).

This indicates that the means of Src_bytes and Dst_bytes significantly differ between DoS and Probe, and between Probe and normal, suggesting these categories have distinct traffic volume characteristics.

## Impact of Protocol Type

In [ ]:
# Create a contingency table
contingency_table = pd.crosstab(df_hyp['protocoltype'], df_hyp['attack_category'] != 'normal')

# Perform the Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Print the results
print(f"Chi-square Statistic: {chi2}")
print(f"P-value: {p}")

if p < 0.05:
    print("Reject the null hypothesis: Significant association between protocol type and anomaly detection.")
else:
    print("Fail to reject the null hypothesis: No significant association between protocol type and anomaly detection.")


## Impact of Flag

In [ ]:
# Assuming df_hyp is your dataframe and 'attack_category' is the target variable

# Prepare the data
df_hyp['is_anomaly'] = np.where(df_hyp['attack_category'] == 'normal', 0, 1)
X = pd.get_dummies(df_hyp['flag'], drop_first=True)
y = df_hyp['is_anomaly']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit the Logistic Regression Model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

# Predict and evaluate
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred))

# Print the coefficients
coefficients = pd.DataFrame(log_reg.coef_.T, index=X.columns, columns=['Coefficient'])
print(coefficients)

# Interpret the significance
logit_model = sm.Logit(y_train, sm.add_constant(X_train))
result = logit_model.fit()
print(result.summary())


Summary of Results:

* Classification Report:

    * Overall Accuracy: 88%

    * Precision, Recall, F1-Score:

        * Class 0 (Normal): High precision (0.84), recall (0.95), and F1-score (0.89).
        * Class 1 (Anomalous): High precision (0.93), recall (0.80), and F1-score (0.86).

This indicates that the model is effective at identifying both normal and anomalous connections, though it is slightly better at identifying normal connections.

* Logistic Regression Coefficients:

    * Positive Coefficients: Flags like S0, SH, RSTR have significant positive coefficients, indicating a higher likelihood of anomalies.

    * Negative Coefficients: Flags like S1, S2, S3, SF have significant negative coefficients, indicating a lower likelihood of anomalies.

* Statistical Significance (p-values):

    * Flags such as RSTO, RSTR, S0, S1, S2, S3, SF, and SH have p-values < 0.05, indicating they are significantly associated with anomalies.
    * Flags REJ and RSTOS0 are not significant (p > 0.05).

# Model Building

## Data Imbalance - SMOTE

In [ ]:
df['attack_category'].value_counts()

In [ ]:
# Apply SMOTE on the attack type '2' and '3'
smote = SMOTE(sampling_strategy={3: 150, 2:1000})
X = df.drop(columns=['attack_category', 'attack', 'type'])
y = df['attack_category']

X_resampled, y_resampled = smote.fit_resample(X, y)

# Convert back to DataFrame
df = pd.DataFrame(X_resampled, columns=X.columns)
df['attack_category'] = y_resampled

In [ ]:
df['attack_category'].value_counts()

## Standardize the Data

In [ ]:
X = df.drop('attack_category', axis=1)
y = df['attack_category']

In [ ]:
X.head()

In [ ]:
col = X.columns

scaler = StandardScaler()
X_std = scaler.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=col)

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

## **Supervised Learning**

In [ ]:
labels = le.classes_

In [ ]:
mlflow.set_experiment('Network Anomaly Detection')

### SVC

In [ ]:
# Start MLflow run
with mlflow.start_run():
    
    # Train the model
    svm = SVC(kernel='sigmoid', random_state=42)
    svm.fit(X_train, y_train)

    # Validation Data
    # Prediction
    y_pred_val = svm.predict(X_val)

    # Set tag
    mlflow.set_tag('mlflow.runName', 'svm_sig')

    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)

    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')

    # Classification Report
    classification_report_val = classification_report(y_val, y_pred_val)
    print('Validation Classification Report:\n', classification_report_val)

    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')

    # Test Data
    # Prediction
    y_pred = svm.predict(X_test)

    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')

    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)

    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log parameters
    mlflow.log_param("kernel", "sigmoid")

    # Log model
    mlflow.sklearn.log_model(svm, "model")

### RandomForest Classification

In [ ]:
with mlflow.start_run():
    # Train the model
    rfc = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
    rfc.fit(X_train, y_train)
    
    # Validation Data
    # Prediction
    y_pred_val = rfc.predict(X_val)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'rf_200_d20')
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = rfc.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(rfc, "model")

    # Log parameters
    mlflow.log_param("n_estimators", rfc.n_estimators)
    mlflow.log_param("max_depth", rfc.max_depth)
    mlflow.log_param("random_state", rfc.random_state)

#### Feature Importance

In [ ]:
# Get feature importances
feature_importances = rfc.feature_importances_

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
})

# Sort the DataFrame by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importances in RandomForestClassifier')
plt.show()

#### Feature Reduction

In [ ]:
df_red = feature_importance_df[feature_importance_df['Importance']<=0.015]
df_red.info()

In [ ]:
cols = df_red['Feature'].tolist()
cols

In [ ]:
X_train_red = X_train.drop(cols, axis=1)
X_val_red = X_val.drop(cols, axis=1)
X_test_red = X_test.drop(cols, axis=1)

In [ ]:
X_train_red.columns

In [ ]:
with mlflow.start_run():
    # Train the model
    rfc = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
    rfc.fit(X_train_red, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'rf_red_200_d20_red21')
    
    # Validation Data
    # Prediction
    y_pred_val = rfc.predict(X_val_red)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = rfc.predict(X_test_red)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(rfc, "model")

    # Log parameters
    mlflow.log_param("n_estimators", rfc.n_estimators)
    mlflow.log_param("max_depth", rfc.max_depth)
    mlflow.log_param("random_state", rfc.random_state)

### GBDT

In [ ]:
with mlflow.start_run():
    # Train the model
    gbt = GradientBoostingClassifier(n_estimators=50, max_depth=20, learning_rate=0.05, random_state=42)
    gbt.fit(X_train, y_train)
    
    # Validation Data
    # Prediction
    y_pred_val = gbt.predict(X_val)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'gbdt_50_d20_l0.05')
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = gbt.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(gbt, "model")

    # Log parameters
    mlflow.log_param("n_estimators", gbt.n_estimators)
    mlflow.log_param("max_depth", gbt.max_depth)
    mlflow.log_param("random_state", gbt.random_state)

### XGBoost

In [ ]:
with mlflow.start_run():
    # Train the model
    xgb = XGBClassifier(n_estimators=250, max_depth=20, learning_rate=0.07, random_state=42)
    xgb.fit(X_train, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'xgb_n250_d20_l.07')
        
    # Validation Data
    # Prediction
    y_pred_val = xgb.predict(X_val)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = xgb.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(xgb, "model")

    # Log parameters
    mlflow.log_param("n_estimators", xgb.n_estimators)
    mlflow.log_param("max_depth", xgb.max_depth)
    mlflow.log_param("learning_rate", xgb.learning_rate)
    mlflow.log_param("random_state", xgb.random_state)

#### Feature Importance

In [ ]:
# Plot feature importance
# Get feature importances
feature_importances = xgb.feature_importances_

# Create a DataFrame
feature_importances_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importances})

# Sort features by importance
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=True)

# Plot
plt.figure(figsize=(20,12))
plt.barh(feature_importances_df['Feature'], feature_importances_df['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.show()

#### Feature Reduction

In [ ]:
with mlflow.start_run():
    # Train the model
    xgb = XGBClassifier(n_estimators=250, max_depth=20, learning_rate=0.07, random_state=42)
    xgb.fit(X_train_red, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'xgb_n250_d20_l.07_red21')
        
    # Validation Data
    # Prediction
    y_pred_val = xgb.predict(X_val_red)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = xgb.predict(X_test_red)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(xgb, "model")

    # Log parameters
    mlflow.log_param("n_estimators", xgb.n_estimators)
    mlflow.log_param("max_depth", xgb.max_depth)
    mlflow.log_param("learning_rate", xgb.learning_rate)
    mlflow.log_param("random_state", xgb.random_state)

### AdaBoost

In [ ]:
with mlflow.start_run():
    # Train the model
    adb = AdaBoostClassifier(random_state=42)
    adb.fit(X_train, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'adb')
    
    # Validation Data
    # Prediction
    y_pred_val = adb.predict(X_val)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = adb.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(adb, "model")

    # Log parameters
    mlflow.log_param("n_estimators", adb.n_estimators)
    mlflow.log_param("random_state", adb.random_state)

### LightGBM

In [ ]:
# Start MLflow run
with mlflow.start_run():
    lgbm = lightgbm.LGBMClassifier(n_estimators=50, max_depth=10, random_state=42)
    lgbm.fit(X_train, y_train)

    # Validation Data
    # Prediction
    y_pred_val = lgbm.predict(X_val)

    # Set tag
    mlflow.set_tag('mlflow.runName', 'lgbm_50_d10')

    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)

    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')

    # Classification Report
    classification_report_val = classification_report(y_val, y_pred_val)
    print('Validation Classification Report:\n', classification_report_val)

    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')

    # Test Data
    # Prediction
    y_pred = lgbm.predict(X_test)

    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')

    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)

    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')

    # Log model
    mlflow.sklearn.log_model(lgbm, "model")
    
    # Log parameters
    mlflow.log_param("n_estimators", lgbm.n_estimators)
    mlflow.log_param("max_depth", lgbm.max_depth)
    mlflow.log_param("learning_rate", lgbm.learning_rate)
    mlflow.log_param("random_state", lgbm.random_state)

### Voting Classifier

#### Without feature reduction

In [ ]:
# Start MLflow run
with mlflow.start_run():
    
    # Create a Voting Classifier
    voting_clf = VotingClassifier(estimators=[('rfc', rfc), ('xgb', xgb)], voting='soft', weights=(2,1))

    # Train the Voting Classifier
    voting_clf.fit(X_train, y_train)
    
    # Set tag
    mlflow.set_tag('mlflow.runName', 'vc_soft_w21')

    # Validation Data
    # Prediction
    y_pred_val = voting_clf.predict(X_val)

    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)

    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')

    # Classification Report
    classification_report_val = classification_report(y_val, y_pred_val)
    print('Validation Classification Report:\n', classification_report_val)

    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')

    # Test Data
    # Prediction
    y_pred = voting_clf.predict(X_test)

    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')

    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)

    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')

    # Log parameters
    mlflow.log_param("voting", voting_clf.voting)

    # Log model
    mlflow.sklearn.log_model(voting_clf, "model")

#### With feature reduction

In [ ]:
# Start MLflow run
with mlflow.start_run():
    
    # Create a Voting Classifier
    voting_clf = VotingClassifier(estimators=[('rfc', rfc), ('xgb', xgb)], voting='soft', weights=(2,1))

    # Train the Voting Classifier
    voting_clf.fit(X_train_red, y_train)
    
    # Set tag
    mlflow.set_tag('mlflow.runName', 'vc_soft_red21_w21')

    # Validation Data
    # Prediction
    y_pred_val = voting_clf.predict(X_val_red)

    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)

    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')

    # Classification Report
    classification_report_val = classification_report(y_val, y_pred_val)
    print('Validation Classification Report:\n', classification_report_val)

    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')

    # Test Data
    # Prediction
    y_pred = voting_clf.predict(X_test_red)

    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')

    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)

    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')

    # Log parameters
    mlflow.log_param("voting", voting_clf.voting)

    # Log model
    mlflow.sklearn.log_model(voting_clf, "model")

#### Probabilities of the prediction

In [ ]:
# Create test dataframe
y_test_df = pd.DataFrame(y_test_transform, columns=['y_test'])
y_pred_df = pd.DataFrame(y_pred_transform, columns=['y_pred'])

# Concatenate X_test, y_test_df, and y_pred_df
df_test = pd.concat([pd.DataFrame(X_test_red).reset_index(drop=True), y_test_df.reset_index(drop=True), y_pred_df.reset_index(drop=True)], axis=1)

In [ ]:
# Predict probabilities
df_test['prob_DoS'] = voting_clf.predict_proba(X_test_red)[:, 0]
df_test['prob_Probe'] = voting_clf.predict_proba(X_test_red)[:, 1]
df_test['prob_R2L'] = voting_clf.predict_proba(X_test_red)[:, 2]
df_test['prob_U2R'] = voting_clf.predict_proba(X_test_red)[:, 3]
df_test['prob_normal'] = voting_clf.predict_proba(X_test_red)[:, 4]

In [ ]:
def prob_outcome(row, probas):
    y_pred = row['y_pred']
    row_index = row.name  # Get the row index to fetch the correct probability
    if y_pred == 'DoS':
        return probas[row_index, 0]
    elif y_pred == 'Probe':
        return probas[row_index, 1]
    elif y_pred == 'R2L':
        return probas[row_index, 2]
    elif y_pred == 'U2R':
        return probas[row_index, 3]
    else:
        return probas[row_index, 4]

probas = voting_clf.predict_proba(X_test_red)
df_test['prob_outcome'] = df_test.apply(lambda row: prob_outcome(row, probas), axis=1)

In [ ]:
df_test.head()

In [ ]:
df_test[df_test['y_test']!=df_test['y_pred']]

### ML flow UI

In [ ]:
!mlflow ui -p 8800

### Final Model Building

#### Standardization of data - 21 features

In [ ]:
cols = df_red['Feature'].tolist()

In [ ]:
X_red = X.drop(cols, axis=1)

col = X_red.columns

scaler = StandardScaler()
X_std = scaler.fit_transform(X_red)
X_std = pd.DataFrame(X_std, columns=col)

In [ ]:
X_std.head()

#### Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_std, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

#### RandomForest Classification

In [ ]:
with mlflow.start_run():
    # Train the model
    rfc = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
    rfc.fit(X_train, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'rf_red_200_d20_red21')
    
    # Validation Data
    # Prediction
    y_pred_val = rfc.predict(X_val)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = rfc.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(rfc, "model")

    # Log parameters
    mlflow.log_param("n_estimators", rfc.n_estimators)
    mlflow.log_param("max_depth", rfc.max_depth)
    mlflow.log_param("random_state", rfc.random_state)

#### XGBoost

In [ ]:
with mlflow.start_run():
    # Train the model
    xgb = XGBClassifier(n_estimators=250, max_depth=20, learning_rate=0.06, random_state=42)
    xgb.fit(X_train, y_train)
    
    # set tag
    mlflow.set_tag('mlflow.runName', 'xgb_n250_d20_l.06_red21')
        
    # Validation Data
    # Prediction
    y_pred_val = xgb.predict(X_val)
    
    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)
    
    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')
    
    # Classification Report
    y_val_transform = le.inverse_transform(y_val)
    y_pred_val_transform = le.inverse_transform(y_pred_val)
    classification_report_val = classification_report(y_val_transform, y_pred_val_transform)
    print('Validation Classification Report:\n', classification_report_val)
    
    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')
    
    # Test Data
    # Prediction
    y_pred = xgb.predict(X_test)
    
    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)
    
    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')
    
    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)
    
    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')
    
    # Log model
    mlflow.sklearn.log_model(xgb, "model")

    # Log parameters
    mlflow.log_param("n_estimators", xgb.n_estimators)
    mlflow.log_param("max_depth", xgb.max_depth)
    mlflow.log_param("learning_rate", xgb.learning_rate)
    mlflow.log_param("random_state", xgb.random_state)

#### Voting Classifier

In [ ]:
# Start MLflow run
with mlflow.start_run():
    
    # Create a Voting Classifier
    voting_clf = VotingClassifier(estimators=[('rfc', rfc), ('xgb', xgb)], voting='soft', weights=(2,1))

    # Train the Voting Classifier
    voting_clf.fit(X_train, y_train)
    
    # Set tag
    mlflow.set_tag('mlflow.runName', 'vc_soft_red21_w21')

    # Validation Data
    # Prediction
    y_pred_val = voting_clf.predict(X_val)

    # Accuracy
    val_acc = accuracy_score(y_val, y_pred_val)
    mlflow.log_metric("val_accuracy", val_acc)

    # Confusion Matrix
    cm_val = confusion_matrix(y_val, y_pred_val)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Validation Confusion Matrix')
    plt.savefig('val_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('val_confusion_matrix.png')

    # Classification Report
    classification_report_val = classification_report(y_val, y_pred_val)
    print('Validation Classification Report:\n', classification_report_val)

    with open('val_classification_report.txt', 'w') as f:
        f.write(classification_report_val)
    mlflow.log_artifact('val_classification_report.txt')

    # Test Data
    # Prediction
    y_pred = voting_clf.predict(X_test)

    # Accuracy
    test_acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", test_acc)

    # Confusion Matrix
    cm_test = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.savefig('test_confusion_matrix.png')
    plt.show()
    mlflow.log_artifact('test_confusion_matrix.png')

    # Classification Report
    y_test_transform = le.inverse_transform(y_test)
    y_pred_transform = le.inverse_transform(y_pred)
    classification_report_test = classification_report(y_test_transform, y_pred_transform)
    print('Test Classification Report:\n', classification_report_test)

    with open('test_classification_report.txt', 'w') as f:
        f.write(classification_report_test)
    mlflow.log_artifact('test_classification_report.txt')

    # Log parameters
    mlflow.log_param("voting", voting_clf.voting)

    # Log model
    mlflow.sklearn.log_model(voting_clf, "model")

### Pickle

In [ ]:
model_path = '/Users/saumyanishi/Documents/GitHub/Cybersecurity/model/model.pkl'

# Save the model
with open(model_path, 'wb') as f:
    pickle.dump((voting_clf, scaler), f)